# Linear Systems Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Gaussian elimination with partial pivoting

In [ ]:
```python

import numpy as np

def gaussian_elimination(A, b):

    n = len(b)

    Ab = np.hstack([A.astype(float), b.reshape(-1, 1).astype(float)])

    for k in range(n):

        max_row = k + np.argmax(np.abs(Ab[k:, k]))

        Ab[[k, max_row]] = Ab[[max_row, k]]

        if abs(Ab[k, k]) < 1e-12:

            raise ValueError(f"Matrix is singular or nearly singular at pivot {k}")

        for i in range(k + 1, n):

            m = Ab[i, k] / Ab[k, k]

            Ab[i, k:] -= m * Ab[k, k:]

    x = np.zeros(n)

    for i in range(n - 1, -1, -1):

        x[i] = (Ab[i, -1] - Ab[i, i+1:n] @ x[i+1:n]) / Ab[i, i]

    return x

In [ ]:
```

### Step 2: LU decomposition

In [ ]:
```python

def lu_decompose(A):

    n = A.shape[0]

    L = np.eye(n)

    U = A.astype(float).copy()

    P = np.eye(n)

    for k in range(n):

        max_row = k + np.argmax(np.abs(U[k:, k]))

        if max_row != k:

            U[[k, max_row]] = U[[max_row, k]]

            P[[k, max_row]] = P[[max_row, k]]

            if k > 0:

                L[[k, max_row], :k] = L[[max_row, k], :k]

        for i in range(k + 1, n):

            L[i, k] = U[i, k] / U[k, k]

            U[i, k:] -= L[i, k] * U[k, k:]

    return P, L, U

def lu_solve(P, L, U, b):

    n = len(b)

    Pb = P @ b.astype(float)

    y = np.zeros(n)

    for i in range(n):

        y[i] = Pb[i] - L[i, :i] @ y[:i]

    x = np.zeros(n)

    for i in range(n - 1, -1, -1):

        x[i] = (y[i] - U[i, i+1:] @ x[i+1:]) / U[i, i]

    return x

In [ ]:
```

### Step 3: Cholesky decomposition

In [ ]:
```python

def cholesky(A):

    n = A.shape[0]

    L = np.zeros_like(A, dtype=float)

    for i in range(n):

        for j in range(i + 1):

            s = A[i, j] - L[i, :j] @ L[j, :j]

            if i == j:

                if s <= 0:

                    raise ValueError("Matrix is not positive definite")

                L[i, j] = np.sqrt(s)

            else:

                L[i, j] = s / L[j, j]

    return L

In [ ]:
```

### Step 4: Least squares via normal equations

In [ ]:
```python

def least_squares_normal(A, b):

    AtA = A.T @ A

    Atb = A.T @ b

    return gaussian_elimination(AtA, Atb)

def ridge_regression(A, b, lam):

    n = A.shape[1]

    AtA = A.T @ A + lam * np.eye(n)

    Atb = A.T @ b

    L = cholesky(AtA)

    y = np.zeros(n)

    for i in range(n):

        y[i] = (Atb[i] - L[i, :i] @ y[:i]) / L[i, i]

    x = np.zeros(n)

    for i in range(n - 1, -1, -1):

        x[i] = (y[i] - L.T[i, i+1:] @ x[i+1:]) / L.T[i, i]

    return x

In [ ]:
```

### Step 5: Condition number

In [ ]:
```python

def condition_number(A):

    U, S, Vt = np.linalg.svd(A)

    return S[0] / S[-1]

In [ ]:
```

## Exercises

In [ ]:
1. Solve the system `[[1,2,3],[4,5,6],[7,8,10]] x = [6, 15, 27]` using your Gaussian elimination, your LU solver, and `np.linalg.solve`. Verify all three give the same answer within floating-point tolerance.

2. Generate a 50x5 random matrix X and target y = X @ w_true + noise. Solve for w using normal equations, QR (via `np.linalg.qr`), SVD (via `np.linalg.svd`), and `np.linalg.lstsq`. Compare all four solutions. Measure the condition number of X^T X and explain how it affects which method you trust.

3. Create a nearly singular matrix by making two columns almost identical (e.g., column 2 = column 1 + 1e-10 * noise). Compute its condition number. Solve Ax = b with and without regularization (add 0.01 * I). Compare the solutions and residuals. Explain why regularization helps.

4. Implement the conjugate gradient algorithm for a 100x100 random symmetric positive definite matrix. Count how many iterations it takes to converge to tolerance 1e-8. Compare with the theoretical maximum of n iterations.

5. Time your Cholesky solver vs your LU solver vs `np.linalg.solve` on symmetric positive definite matrices of size 10, 50, 200, 500. Plot the results. Verify Cholesky is roughly 2x faster than LU.